# Spark DataFrames, Execution Plans & Partitioning

**Notebook:** `spark-dataframe-execution-plans.ipynb`  
**Dataset:** NYC Motor Vehicle Collisions — frozen snapshot (`nyc_crashes_snapshot.parquet`)  
**Execution Platform:** Google Colab  

Your course handout gives the marks for each question, how to submit, and which track you complete.

---

### What Do I Complete?
| Part | Covers | Items |
| :--- | :--- | :--- |
| **Core** (everyone) | Phase 1: Load & Inspect · Phase 2: Exploratory Analysis · Phase 3: Plans, Shuffles & AQE | Q01–Q11, S01 |
| **Track A: Laziness & Dependencies** | Narrow vs wide transformations, lazy evaluation | Q12–Q13 |
| **Track B: MapReduce & Partition Skew** | The RDD API as MapReduce, hash partitioning, task telemetry | Q14–Q16, C01, E01 |

Complete the **Core**, then the track your handout assigns.

### Execution Platform
Run this notebook in **Google Colab** (open it via Colab's GitHub tab, or upload it). The setup cells install a **pinned** PySpark version and download the frozen dataset snapshot.
* **Do not change the pinned PySpark version or the `SparkSession` configuration.** Physical plans, partition counts and job counts differ between Spark releases and settings; the expected answers assume the configuration provided here.
* **Never commit the dataset** (`*.parquet`) and **never commit cell outputs**; export your evidence PDF **before** clearing outputs (see the final section).

## 0. Environment Setup

Install the pinned PySpark release and confirm that Java is available. Colab ships with a Java runtime; the cell installs one only if it is missing.

In [ ]:
# Pinned versions: DO NOT CHANGE (execution plans differ between Spark releases)
PYSPARK_VERSION = "3.5.3"

import shutil
if shutil.which("java") is None:
    !apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!pip install -q pyspark=={PYSPARK_VERSION}

import pyspark
print(f"PySpark version: {pyspark.__version__}")
assert pyspark.__version__ == PYSPARK_VERSION, (
    f"❌ Expected PySpark {PYSPARK_VERSION} but found {pyspark.__version__}. "
    "Restart the runtime (Runtime -> Restart session) and re-run this cell."
)
!java -version 2>&1 | head -n 1

### Download the Frozen Dataset Snapshot

The live NYC dataset changes daily, so the course provides a **frozen snapshot** so that everyone's answers are comparable. The cell below downloads it (once) and verifies its checksum.

In [ ]:
import os
import hashlib

SNAPSHOT_FILE = "nyc_crashes_snapshot.parquet"
SNAPSHOT_URL = "https://github.com/jackdotwa/us-ie-big-data-technologies/releases/download/v2026.1.0-nyc/nyc_crashes_snapshot.parquet"
SNAPSHOT_SHA256 = "d8c431176429d6cb773bb97f7fb78e1710516dbda7fef135fd8261422a3666c4"

if not os.path.exists(SNAPSHOT_FILE):
    print(f"ℹ '{SNAPSHOT_FILE}' not found locally. Downloading course snapshot...")
    !wget -q -O {SNAPSHOT_FILE} {SNAPSHOT_URL}

if not os.path.exists(SNAPSHOT_FILE) or os.path.getsize(SNAPSHOT_FILE) == 0:
    raise FileNotFoundError(
        "\n====================================================================================\n"
        f"❌ MISSING FILE: '{SNAPSHOT_FILE}' could not be found or downloaded!\n"
        "👉 ACTION REQUIRED: Upload the snapshot to Google Colab using the Files sidebar\n"
        "   (folder icon on the left panel) before running the Phase 1 cells.\n"
        "===================================================================================="
    )

if SNAPSHOT_SHA256:
    h = hashlib.sha256()
    with open(SNAPSHOT_FILE, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == SNAPSHOT_SHA256, "❌ Snapshot checksum mismatch: delete the file and re-run this cell."
    print("✅ Snapshot checksum verified.")

print(f"✅ {SNAPSHOT_FILE}: {os.path.getsize(SNAPSHOT_FILE) / 1e6:.1f} MB")

## Phase 1: Load & Inspect

The `SparkSession` below is configured for you. It pins the number of local cores to 2 (Colab's free-tier CPU count) so that partition counts are the same for everyone, and leaves the shuffle-partition and AQE settings at Spark's defaults.

Then load the snapshot with `spark.read.parquet(...)` and inspect it with `printSchema()` and `count()`.

* **Q01:** Exact total number of rows in the snapshot.
* **Q02:** The Spark data type `printSchema()` reports for the `CRASH DATE` column.
* **S01:** Screenshot of your `printSchema()` result and row count.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# DO NOT CHANGE: the expected answers assume this configuration
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("spark-dataframe-execution-plans")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "200")   # Spark default
    .config("spark.sql.adaptive.enabled", "true")    # Spark default
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
sc = spark.sparkContext
print(f"Spark {spark.version} | application ID: {sc.applicationId} | UI/REST API: {sc.uiWebUrl}")

In [ ]:
# 1.1 Load the snapshot into a DataFrame named `df`
# TODO: df = spark.read.parquet(...)
df = None

In [ ]:
# 1.2 Print the schema (Q02)
# TODO

In [ ]:
# 1.3 Count the rows (Q01)
# TODO

## Phase 2: Exploratory Analysis with the Spark DataFrame API

Answer each question by writing **Spark DataFrame** code in the cell provided and running it on the **full** snapshot.

**Inspect before you aggregate.** Real operational data is messy. Before answering, look at the columns involved: how many values are null, which placeholder values appear, and whether the same thing is recorded in more than one way. Useful tools:
* `df.select(col).distinct()`
* `df.groupBy(col).count().orderBy(F.desc("count"))`
* `F.col(c).isNull()`
* `df.describe()`

A query that runs without error can still give the wrong answer.

**Rule:** Use the DataFrame API only (`select`, `filter`, `groupBy`, `agg`, `orderBy`, functions from `pyspark.sql.functions`). Do **not** use `spark.sql(...)`.

*Tip:* column names contain spaces, so refer to them with `F.col("CRASH DATE")` rather than attribute access.

In [ ]:
# 2.0 Data inspection scratch space: profile the columns you will use in Q03–Q08
# (nulls, placeholder values, inconsistent spellings). Add more cells as needed.
# TODO

In [ ]:
# Q03: On which calendar date did the most crashes occur, and how many crashes that day?
# Answer format: YYYY-MM-DD, <count>
# TODO

In [ ]:
# Q04: On which day of the week do the most crashes occur?
# Hint: F.date_format(col, "EEEE")
# TODO

In [ ]:
# Q05: Which NAMED NYC borough has the most crashes?
# TODO

In [ ]:
# Q06: What percentage of all crashes have no borough recorded (BOROUGH is null)?
# Round to 2 decimal places.
# TODO

In [ ]:
# Q07: In QUEENS, what is the most common contributing factor actually IDENTIFIED
# in `CONTRIBUTING FACTOR VEHICLE 1`? (A placeholder meaning "not recorded" is not a factor.)
# Assign the query DataFrame to `q07_df`: you will call .explain() on it in Q09.
# TODO: q07_df = df.filter(...)...
q07_df = None

In [ ]:
# Q08: For each (VEHICLE TYPE CODE 1, BOROUGH) combination (both non-null, values as recorded)
# with at least 1,000 crashes, compute the average NUMBER OF PERSONS INJURED.
# Which combination has the highest average, and what is it?
# Answer format: <vehicle type>, <borough>, <average to 3 decimals>
# TODO

## Phase 3: Under the Hood — Plans, Shuffles & Laziness

**Core:** everyone answers Q09–Q11, then continues with the track their handout assigns: **Track A** (Q12–Q13) or **Track B** (Q14–Q16).

Every Spark DataFrame query is turned into a **physical plan** before it runs. Print it with `.explain()` and read it from the **bottom up**: the bottom is the file scan, the top is the final result.

* An `Exchange` operator is a **shuffle**: rows are redistributed across partitions so that matching keys end up together. This is the MapReduce shuffle, planned for you.
* **Narrow** transformations (each output partition depends on one input partition) need no shuffle. **Wide** transformations need an `Exchange`.
* `PushedFilters` and `ReadSchema` on the `FileScan` line show what Spark avoided reading from Parquet.
* **Adaptive Query Execution (AQE)** re-optimises a query while it runs, using real statistics from completed stages.

In [ ]:
# Q09: Call .explain() on your Q07 query (`q07_df`).
# (a) How many `HashAggregate` operators does the physical plan contain?
# (b) The `HashAggregate` BELOW the `Exchange` computes `partial_count`.
#     Which MapReduce component does it correspond to? (Mapper / Combiner / Partitioner / Reducer)
# TODO

In [ ]:
# Q10: Call .explain() on the query below.
# (a) List the PushedFilters. (b) How many of the 29 columns appear in ReadSchema?
# Answer format: <filters>; <n>
q10_df = df.select("COLLISION_ID", "ZIP CODE").filter(F.col("ZIP CODE") == "11201")
# TODO

In [ ]:
# Q11: Run df.groupBy("BOROUGH").count() and inspect .rdd.getNumPartitions() of the result,
# first with AQE disabled, then with AQE enabled.
# Answer format: <partitions with AQE off>, <partitions with AQE on>
spark.conf.set("spark.sql.adaptive.enabled", "false")
# TODO: number of partitions with AQE off

spark.conf.set("spark.sql.adaptive.enabled", "true")
# TODO: number of partitions with AQE on

## Track A: Laziness & Dependencies

*Complete this track if your handout assigns it.*

### Q12: Narrow or Wide?
Classify each of `filter`, `select`, `withColumn`, `groupBy().agg()`, `orderBy` and `join` (non-broadcast) as **narrow** or **wide**. You may use `.explain()` on small examples to check your reasoning (look for `Exchange`). Record your classification below and submit it as your handout instructs.

| Transformation | Narrow / Wide |
| :--- | :--- |
| `filter` | |
| `select` | |
| `withColumn` | |
| `groupBy().agg()` | |
| `orderBy` | |
| `join` (non-broadcast) | |

In [ ]:
# Q12 (optional scratch space): check your classification with .explain()
# TODO

### Q13: Lazy Evaluation
Spark **transformations** only build a plan; **actions** run it as one or more Spark **jobs**. The provided `count_jobs(...)` helper runs a function and reports how many Spark jobs it executed. The first example shows it counting an action.

In [ ]:
# PROVIDED HELPER: do not modify
import time
import uuid


def count_jobs(fn, label="count_jobs"):
    # Runs fn() and returns (result, number of Spark jobs fn executed).
    group = f"{label}-{uuid.uuid4().hex[:8]}"
    sc.setJobGroup(group, label)
    try:
        result = fn()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
        sc.setLocalProperty("spark.job.description", None)
    time.sleep(1.0)  # let Spark's listener bus register the finished jobs
    n_jobs = len(sc.statusTracker().getJobIdsForGroup(group))
    print(f"🔎 Spark jobs executed: {n_jobs}")
    return result, n_jobs


# Example: an ACTION runs at least one job
_ = count_jobs(lambda: df.count())

In [ ]:
# Q13: Using count_jobs(...), run ONLY the transformations filter, select and withColumn on df.
# How many Spark jobs were executed?
# Build the transformations on the existing `df`: do not re-read the file inside count_jobs.
def transformations_only():
    # TODO: return df.filter(...).select(...).withColumn(...)
    pass

_ = count_jobs(transformations_only)

## Track B: MapReduce & Partition Skew

*Complete this track if your handout assigns it.*

### Primer: Lambdas & the RDD API (not graded)
The DataFrame API takes column expressions; the lower-level **RDD API** takes plain Python **functions**, one call per record. Those functions are usually written as **lambdas**: small anonymous functions, `lambda <args>: <expression>`.

In [ ]:
# A lambda is a function without a name: these two are equivalent
def square(x):
    return x * x

square_lambda = lambda x: x * x
print(square(4), square_lambda(4))

# Lambdas are mostly passed straight into other functions
pairs = [("b", 2), ("a", 5), ("c", 1)]
print(list(map(lambda x: x * 10, [1, 2, 3])))      # apply to every element
print(sorted(pairs, key=lambda kv: -kv[1]))         # sort by the second item, descending

An RDD is a distributed collection of Python objects, split into **partitions**. `map` applies a function to every record (narrow: no shuffle), and `reduceByKey` merges the values of `(key, value)` pairs with a two-argument function (wide: shuffles by key). Below is the classic MapReduce word count on a 2-partition RDD.

In [ ]:
lines = sc.parallelize(["the cat sat", "the dog sat", "the end"], numSlices=2)
print("partitions:", lines.glom().collect())   # glom() shows the records in each partition

word_counts = (
    lines
    .flatMap(lambda line: line.split())        # map: one record per word
    .map(lambda word: (word, 1))               # map: emit (key, 1)
    .reduceByKey(lambda a, b: a + b)           # shuffle + reduce: sum per key
)
print(sorted(word_counts.collect(), key=lambda kv: -kv[1]))

`reduceByKey` applies your function **within each partition before the shuffle** (exactly the combiner, i.e. the `partial_count` you found in Q09), so only one pair per key per partition is sent. `groupByKey` ships every single pair and should be avoided for aggregations.

A DataFrame can be converted to an RDD with `.rdd`; each record is then a `Row`, whose fields you read by name (column names may contain spaces) or by position:

In [ ]:
rows = df.select("COLLISION_ID", "CRASH DATE").rdd
first = rows.first()
print(type(first).__name__, first)
print(first["CRASH DATE"], first[0])

### Task B.1: The Same Aggregation as MapReduce
Re-implement Question 05 using the RDD API in classic MapReduce form. Your function must use `map` and `reduceByKey` (no DataFrame aggregations) and return an RDD of `(borough, count)` pairs.

**Keep the function name and signature exactly as given, and keep the function in its own cell** (C01: an automated test runner imports and executes it against test DataFrame fixtures).

In [ ]:
def mapreduce_borough_counts(df):
    """
    Implements borough crash aggregation using the Spark RDD MapReduce paradigm.
    Returns an RDD of (borough, count) pairs.
    """
    # TODO: implement with the RDD API only: use map and reduceByKey
    #       (no DataFrame groupBy/agg). Return an RDD of (borough, count) pairs.
    raise NotImplementedError

In [ ]:
# Q14: What count does your MapReduce job produce for BROOKLYN?
try:
    borough_counts = mapreduce_borough_counts(df).collect()
    for borough, n in sorted(borough_counts, key=lambda kv: -kv[1]):
        print(f"{str(borough):<15} {n:>10,}")
except NotImplementedError:
    print("Implement mapreduce_borough_counts() first.")

### Task B.2: Diagnosing Skew
Spark decides which partition a row goes to by hashing its key (`hash(key) % numPartitions`). `repartition` is a transformation, so nothing runs until an action is called; the `noop` write is an action that executes the full shuffle without saving any output. Run the cell below unchanged.

In [ ]:
# PROVIDED: run unchanged
sc = spark.sparkContext
sc.setJobDescription("skew")          # labels the stages of the next action
repartitioned_df = df.repartition(6, "BOROUGH")
repartitioned_df.write.format("noop").mode("overwrite").save()   # action: forces the shuffle to run
sc.setJobDescription(None)

**Cell A: Partition Inspection.** Group `repartitioned_df` by `F.spark_partition_id()` to reveal the row count and the specific boroughs allocated to each of the 6 partitions. Remember that a partition with no rows will not appear in a `groupBy` result.

* **Q15:** How many rows are in the largest partition, and how many of the 6 partitions are empty? Format: `<rows>, <empty partitions>`.

In [ ]:
# Cell A: rows and boroughs per partition
# TODO

**Cell B: Spark REST API Telemetry.** In Colab, the Spark Web UI (port 4040) is not reachable from your browser, so query Spark's REST API from Python instead. Live telemetry is served under `http://localhost:4040/api/v1/applications/{app_id}/`:
* `.../stages` lists every stage with its `stageId`, `attemptId`, `numTasks` and `description`.
* `.../stages/{stage_id}/{attempt_id}/taskList` lists the individual tasks of one stage with their metrics.

Write Python code (using `requests` or `urllib.request`) to:
1. Retrieve your active application ID via `spark.sparkContext.applicationId`.
2. Find the stage that **reads** the shuffle: the stage whose `description` is `skew` and that has 6 tasks.
3. Query its `taskList` and display, for each of the 6 tasks, the task duration and the shuffle records read (`taskMetrics.shuffleReadMetrics.recordsRead`).

* **E01:** Screenshot of this cell's code and its output table (6 task durations, shuffle records read, and your application ID).

In [ ]:
# Cell B: query the Spark REST API for the shuffle-read stage of the "skew" job
import requests

app_id = spark.sparkContext.applicationId
ui_port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[1]   # 4040 unless that port was already taken
base_url = f"http://localhost:{ui_port}/api/v1/applications/{app_id}"
print(f"Application ID: {app_id}")

# TODO: 1. GET {base_url}/stages and find the stage with description "skew" and numTasks == 6
# TODO: 2. GET {base_url}/stages/{stage_id}/{attempt_id}/taskList
# TODO: 3. Display duration and taskMetrics.shuffleReadMetrics.recordsRead for each task

**Q16 (max 120 words):** Using your empirical evidence from Cells A and B, explain (a) why six boroughs produced fewer than six non-empty partitions, (b) which keys share the largest partition and what this does to the stage's wall-clock time, and (c) one concrete change that would balance the work. Draft your answer below and submit it as your handout instructs.

*Your answer:*

## Evidence & Version-Control Hygiene

### 1. Export the evidence PDF **first**
The executed PDF is your execution record. Every code cell and its output (schema, row count, query results, `.explain()` plans, REST API tables) **must be visible**.
1. In Google Colab, select **File -> Print -> Save as PDF**.
2. Confirm that output tables, execution plans and API responses are readable and not truncated.
3. Name the PDF as your handout specifies.

### 2. Clear outputs, then commit
1. Select **Edit -> Clear all outputs**, then save the notebook into your repository.
2. Keep the dataset out of Git: add `*.parquet` to your `.gitignore` before committing.

Repository, branch, merge-request and submission details are in your course handout.

In [ ]:
# Before committing, check locally: git status && git log -n 5 --oneline
# Confirm that no *.parquet file is tracked and the committed notebook has no outputs.
print("Evidence PDF exported? Outputs cleared? Then commit.")